# conformal-nlp Quickstart
A lightweight conformal wrapper for HuggingFace classifiers.
Instead of a point prediction with unreliable confidence, you get a **prediction set**
that contains the true label at least 90% of the time — provably.

In [10]:
import sys
sys.path.insert(0, "..")  # so Python can find conformal_nlp/

from conformal_nlp import ConformalClassifier
from datasets import load_dataset
import numpy as np

print("Imports OK")

Imports OK


In [11]:
# Load SST-2 — binary sentiment (POSITIVE / NEGATIVE)
# We use the validation set since test labels aren't public
dataset = load_dataset("glue", "sst2", split="validation")

# Shuffle for reproducibility
dataset = dataset.shuffle(seed=42)

# Split: 50% calibration, 50% test
n = len(dataset)
n_cal = n // 2

cal_texts  = dataset["sentence"][:n_cal]
cal_labels = dataset["label"][:n_cal]
test_texts  = dataset["sentence"][n_cal:]
test_labels = dataset["label"][n_cal:]

print(f"Calibration set: {len(cal_texts)} examples")
print(f"Test set:        {len(test_texts)} examples")
print(f"\nExample text:  {cal_texts[0]}")
print(f"Example label: {cal_labels[0]}  (0=negative, 1=positive)")

Calibration set: 436 examples
Test set:        436 examples

Example text:  it gets onto the screen just about as much of the novella as one could reasonably expect , and is engrossing and moving in its own right . 
Example label: 1  (0=negative, 1=positive)


In [12]:
# Load a DistilBERT model fine-tuned on SST-2
# This will download ~250MB the first time, then cache it
cc = ConformalClassifier(
    model_name="distilbert-base-uncased-finetuned-sst-2-english",
    score="LAC",
    alpha=0.1,  # target 90% coverage
)

# Calibrate on the held-out calibration set
cc.calibrate(cal_texts, cal_labels)

print(cc)

Loading model 'distilbert-base-uncased-finetuned-sst-2-english' on cpu...
Calibrated | score=LAC | alpha=0.1 | n_cal=436 | q_hat=0.4143
ConformalClassifier(model='distilbert-base-uncased-finetuned-sst-2-english', score=LAC, alpha=0.1, q_hat=0.4143)


In [13]:
# Predict on the test set — returns sets of label strings
prediction_sets = cc.predict(test_texts)

# Evaluate coverage and efficiency
results = cc.evaluate(test_texts, test_labels)

print(f"Empirical coverage:  {results['coverage']:.3f}  (target: {1 - results['alpha']:.1f})")
print(f"Average set size:    {results['avg_set_size']:.3f}  (out of 2 possible labels)")
print(f"q_hat:               {results['q_hat']:.4f}")

# Show a few example prediction sets
print("\n--- Example Predictions ---")
for i in range(5):
    print(f"Text:  {test_texts[i][:60]}...")
    print(f"Set:   {prediction_sets[i]}")
    print(f"True:  {cc.id2label[test_labels[i]]}")
    print()

Empirical coverage:  0.908  (target: 0.9)
Average set size:    0.993  (out of 2 possible labels)
q_hat:               0.4143

--- Example Predictions ---
Text:  the action switches between past and present , but the mater...
Set:   {'NEGATIVE'}
True:  NEGATIVE

Text:  no way i can believe this load of junk . ...
Set:   {'NEGATIVE'}
True:  NEGATIVE

Text:  it made me want to wrench my eyes out of my head and toss th...
Set:   {'NEGATIVE'}
True:  NEGATIVE

Text:  it 's as if you 're watching a movie that was made in 1978 b...
Set:   {'NEGATIVE'}
True:  NEGATIVE

Text:  minority report is exactly what the title indicates , a repo...
Set:   {'NEGATIVE'}
True:  POSITIVE



In [14]:
# Now try RAPS — on binary SST-2 the difference will be small
# but this sets up the pattern for GoEmotions (28 classes)
cc_raps = ConformalClassifier(
    model_name="distilbert-base-uncased-finetuned-sst-2-english",
    score="RAPS",
    alpha=0.1,
)
cc_raps.calibrate(cal_texts, cal_labels)
results_raps = cc_raps.evaluate(test_texts, test_labels)

print("Score comparison on SST-2:")
print(f"{'Score':<8} {'Coverage':<12} {'Avg Set Size':<15}")
print("-" * 35)
print(f"{'LAC':<8} {results['coverage']:<12.3f} {results['avg_set_size']:<15.3f}")
print(f"{'RAPS':<8} {results_raps['coverage']:<12.3f} {results_raps['avg_set_size']:<15.3f}")

# Find examples where the model is uncertain (set size = 2)
pred_sets_raps = cc_raps.predict(test_texts)
uncertain = [
    (test_texts[i], pred_sets_raps[i], cc_raps.id2label[test_labels[i]])
    for i in range(len(test_texts))
    if len(pred_sets_raps[i]) == 2
]
print(f"\nUncertain predictions (set size=2): {len(uncertain)}")
if uncertain:
    print("\nExample uncertain cases:")
    for text, s, true in uncertain[:3]:
        print(f"  Text: {text[:70]}...")
        print(f"  Set: {s}  |  True: {true}")
        print()

Loading model 'distilbert-base-uncased-finetuned-sst-2-english' on cpu...
Calibrated | score=RAPS | alpha=0.1 | n_cal=436 | q_hat=0.9999
Score comparison on SST-2:
Score    Coverage     Avg Set Size   
-----------------------------------
LAC      0.908        0.993          
RAPS     1.000        1.993          

Uncertain predictions (set size=2): 433

Example uncertain cases:
  Text: the action switches between past and present , but the material link i...
  Set: {'NEGATIVE', 'POSITIVE'}  |  True: NEGATIVE

  Text: no way i can believe this load of junk . ...
  Set: {'NEGATIVE', 'POSITIVE'}  |  True: NEGATIVE

  Text: it made me want to wrench my eyes out of my head and toss them at the ...
  Set: {'NEGATIVE', 'POSITIVE'}  |  True: NEGATIVE



## Takeaway: Score choice matters

| Score | Coverage | Avg Set Size | Notes |
|-------|----------|--------------|-------|
| LAC   | 0.908    | 0.993        | Tight, efficient — right tool for binary |
| RAPS  | 1.000    | 1.993        | Overcautious on binary — designed for many-class |

**LAC is the right choice for SST-2.** RAPS will shine on GoEmotions (28 classes)
where LAC produces bloated sets and RAPS's regularization earns its keep.
This is the core benchmarking story for Week 2.

In [15]:
# Summary
print("conformal-nlp on SST-2")
print(f"Model:    distilbert-base-uncased-finetuned-sst-2-english")
print(f"Score:    LAC")
print(f"Alpha:    0.1  (90% coverage target)")
print(f"Coverage: {results['coverage']:.3f}  ✓ guarantee holds")
print(f"Avg set:  {results['avg_set_size']:.3f}  (near-perfect efficiency on binary task)")

conformal-nlp on SST-2
Model:    distilbert-base-uncased-finetuned-sst-2-english
Score:    LAC
Alpha:    0.1  (90% coverage target)
Coverage: 0.908  ✓ guarantee holds
Avg set:  0.993  (near-perfect efficiency on binary task)
